# Hello World: Quantum Computing with Perceval

Welcome! This notebook demonstrates how to use Perceval to simulate and run photonic quantum experiments, both locally and remotely on Quandela's Quantum Hardware. Follow along to learn the basics and try it yourself!

## 1. Install and Import Dependencies

First, let's make sure all required packages are installed and import them.  
If you haven't installed Perceval yet, run:  
`pip install perceval-quandela` in your terminal or `!pip install perceval-quandela` in your notebook.

In [ ]:
### Run this cell to install the perceval-quandela package
!pip install perceval-quandela

## 2. Check Perceval Version

Let's check which version of Perceval is installed to ensure compatibility. The most up to date version is 1.1.0.

In [ ]:
import perceval as pcvl
from perceval.algorithm import Sampler
from tqdm.notebook import tqdm
import time
from pprint import pprint
pcvl.__version__

## 3. Build and Simulate a Simple Quantum Circuit

We'll create a basic quantum circuit using one beam splitter and simulate it locally.  
This helps us understand the basics before moving to the cloud.

In [ ]:
## Define the input state
input_state = pcvl.BasicState('|1,1>')

## Define the circuit
circuit = pcvl.Circuit(2)
circuit.add(0, pcvl.BS())

## Set the number of samples to generate
nsamples = 10_000

## Define a noise model (optional)
#noise_model = pcvl.NoiseModel(brightness=1,indistinguishability=1, g2=0) # uncomment for a noisy simulation

## Build the experiment
experiment = pcvl.Experiment(m_circuit=circuit) 
experiment.with_input(input_state)
experiment.min_detected_photons_filter(1)
#experiment.noise = noise_model  # uncomment for a noisy simulation

## Create a processor to run the experiment
processor = pcvl.Processor("SLOS", experiment)

## Create a sampler to generate samples and compute probabilities
sampler = Sampler(processor)
samples = sampler.sample_count(nsamples)['results']
probs = sampler.probs()['results']  # Ask for the exact probabilities

# Display the results
print("Samples:")
pcvl.pdisplay(samples)
print("Exact probabilities:")
pcvl.pdisplay(probs) 

## 4. Configure Cloud Access

To run jobs on Quandela's real quantum hardware, you'll need an API token from the Quandela Cloud.

1. Go to [Quandela Cloud](https://cloud.quandela.com) - create a free account if you don't have one already.
2. Go to the Cloud Dashboard and create a new Token.

![Token Page Screenshot](../img/CloudToken.png)

3. Copy the Token in the next cell, keeping the quotation marks.
4. Run the next 2 code cells.

In [ ]:
CLOUD_TOKEN = "YOUR_CLOUD_TOKEN"  # Replace with your Quandela Cloud token
## (OPTIONAL) 
# Save your token in a file called .env in the same directory as this notebook
# Then run with the following code:
#from dotenv import load_dotenv
#load_dotenv()
#import os
#CLOUD_TOKEN = os.getenv("CLOUD_TOKEN")

In [ ]:
from perceval import RemoteConfig, RemoteProcessor
# Save your token and proxy configuration into Perceval persistent data, you only need to do it once per machine.
remote_config = RemoteConfig()
remote_config.set_token(CLOUD_TOKEN)  # Replace with your Quandela token which you can create in you Quandela account at https://www.cloud.quandela.com/
remote_config.save()

## 5. Run a Job on Quandela's Quantum Computer

Now, let's submit our circuit to the cloud and monitor the job's progress.

If you change the platform from sim:belenos to qpu:belenos the job will be send not to the simulator anymore but to the quantum computer.

In [ ]:
remote_processor = RemoteProcessor("qpu:belenos").add(0, experiment) # choose between qpu:belenos and sim:belenos
remote_processor.with_input(input_state)
remote_processor.min_detected_photons_filter(1)

remote_sampler = Sampler(remote_processor, max_shots_per_call=1e6)
remote_sampler.default_job_name = "Hello World" 
remote_job = remote_sampler.sample_count.execute_async(nsamples)
print("Job submitted to Quandela Cloud.")
print("Job ID:", remote_job.id)

## 6. Monitor Job Progress

You can see the Job you have just created on Quandela Cloud in the Job tap on the side bar menu.

![Cloud Job Status](../img/JobStatus.png)

In [ ]:
## Check the status of the job
print(f"Job status = {remote_job.status()}")

#### The Job can have one of the following Status
- Waiting -> Job is waiting to be processed
- Running -> Job is actively in progress
- Completed -> Job completed successfully

In [ ]:
### Wait for the job to complete and display a progress bar, run this cell only once the job status is "RUNNING".
previous_prog = 0
with tqdm(total=1, bar_format='{desc}{percentage:3.0f}%|{bar}|') as tq:
    tq.set_description(f'Get {nsamples} samples from {remote_processor.name}')
    while not remote_job.is_complete:
        tq.update(remote_job.status.progress/100-previous_prog)
        previous_prog = remote_job.status.progress/100
        time.sleep(1)
    tq.update(1-previous_prog)
    tq.close()

print(f"Job status = {remote_job.status()}")

## 7. Retrieve and Display Results

Once the job is complete, let's fetch and visualize the results from the quantum hardware. 

In [ ]:
## Get the results of the job and display them
results = remote_job.get_results()
pcvl.pdisplay(results['results'])

If your job was waiting in the queue for a longer time and you closed this notebook, you can still retrieve your results from the Job ID and the Platform Name which you find on Quandela Cloud.

In [ ]:
## Connect to Quandela Cloud
job_id = "JOB_ID"  # Replace with your job id
remote_processor = RemoteProcessor("PLATFORM NAME")  # Replace with the platform you used to run the job, e.g. "sim:slos" or "qpu:belenos"
remote_job = remote_processor.resume_job(job_id) # even though it says resume job this also works for fetching an already completed job

## Retrieve the results of the job and display them
results = remote_job.get_results()   
pcvl.pdisplay(results['results'])

## 8. Extra Tools

You can visualize your circuit with Perceval:

In [ ]:
pcvl.pdisplay(circuit)

You can get the current performance parameters of our QPUs:
- The Clock tells you how many photons are generated per second. (5 MHz means5 million photons per second)
- The Transmittance tells you how likely it is that a photon generated is also detected in the end and not lost on in the process.
- The Status tells you if the QPU is currently available, running, in maintenance or being calibrated.

In [ ]:
belenos = pcvl.RemoteProcessor("qpu:belenos")
perf_belenos = belenos.performance
pprint(f"The Clock Speed of {belenos.name} is: {perf_belenos['Clock (MHz)']} MHz")
pprint(f"The Transmittance of {belenos.name} is: {perf_belenos['Transmittance (%)']} %")
pprint(f"The Status of {belenos.name} is: {belenos.status}")

You can see the physical architecture of the photonic chip.

In [ ]:
specs = belenos.specs
pcvl.pdisplay(specs.architecture, recursive=True)


You can see the constraints of the platform you are using (e.g. number of photons).

In [ ]:
print("Platform constraints:")
pprint(specs.constraints)

# Conclusion

Congratulations! You've simulated and run a quantum experiment on a real quantum computer using Perceval and Quandela Cloud.  
Feel free to experiment with different circuits and parameters!